In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

device = "mps" if torch.backends.mps.is_available() else "cpu"

print("Device:", device)

Device: mps


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer.model_max_length)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

1000000000000000019884624838656


In [3]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)

print("Model loaded")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded


In [5]:
from datasets import load_from_disk

DATASET_PATH = "/Volumes/Hardik/Arxiv_dataset/arxiv_50k_cleaned"

dataset = load_from_disk(DATASET_PATH)

train = dataset["train"]
validation = dataset["validation"]
test = dataset["test"]

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Train: 44971
Validation: 2500
Test: 2500


In [6]:
text = train[0]["article"][:4000]

inputs = tokenizer(
    text,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_length=150,
        min_length=50,
        num_beams=4,
        early_stopping=True
    )

summary = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(summary)

arp 220 is the nearest example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels. it contains two nuclei separated by 350 pc, both surrounded by massive discs of dense molecular gas. radio detections of supernovae at a rate of 13 yr [MATH] confirm that huge populations of massive stars are present.


In [7]:
def chunk_text(text, chunk_size=900, overlap=100):
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []

    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunks.append(tokens[start:end])

        start += chunk_size - overlap

    return chunks

In [8]:
text = train[0]["article"]

chunks = chunk_text(text)

print("Number of chunks:", len(chunks))
print("First chunk tokens:", len(chunks[0]))
print("Last chunk tokens:", len(chunks[-1]))

Number of chunks: 8
First chunk tokens: 900
Last chunk tokens: 736


In [9]:
chunk_text_decoded = tokenizer.decode(
    chunks[0],
    skip_special_tokens=True
)

inputs = tokenizer(
    chunk_text_decoded,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_length=150,
        min_length=50,
        num_beams=4,
        early_stopping=True
    )

summary = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(summary)

arp 220 is the nearest example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels. it contains two nuclei separated by 350 pc, both surrounded by massive discs of dense molecular gas. radio detections of supernovae at a rate of 13 yr [MATH] confirm that huge populations of massive stars are present.


In [10]:
def summarize_long_document(text):
    chunks = chunk_text(text)

    summaries = []

    for chunk in chunks:
        chunk_text_decoded = tokenizer.decode(
            chunk,
            skip_special_tokens=True
        )

        inputs = tokenizer(
            chunk_text_decoded,
            max_length=1024,
            truncation=True,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_length=150,
                min_length=50,
                num_beams=4,
                early_stopping=True
            )

        summary = tokenizer.decode(
            output[0],
            skip_special_tokens=True
        )

        summaries.append(summary)

    combined = " ".join(summaries)

    inputs = tokenizer(
        combined,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=200,
            min_length=100,
            num_beams=4,
            early_stopping=True
        )

    final_summary = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return final_summary

In [12]:
def summarize_chunk(chunk):
    text = tokenizer.decode(chunk, skip_special_tokens=True)

    inputs = tokenizer(
        text,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=150,
            min_length=50,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [13]:
def summarize_long_document(text):
    chunks = chunk_text(text)
    summaries = [summarize_chunk(chunk) for chunk in chunks]

    combined = " ".join(summaries)

    while len(tokenizer.encode(combined, add_special_tokens=False)) > 900:
        chunks = chunk_text(combined)
        summaries = [summarize_chunk(chunk) for chunk in chunks]
        combined = " ".join(summaries)

    inputs = tokenizer(
        combined,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=200,
            min_length=100,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [38]:
def summarize_long_document(text):
    chunks = chunk_text(text)
    chunks = filter_reference_chunks(chunks)

    summaries = []

    for chunk in chunks:
        summaries.append(summarize_chunk(chunk))

    combined = " ".join(summaries)

    while len(tokenizer.encode(combined, add_special_tokens=False)) > 900:
        chunks = chunk_text(combined)
        summaries = [summarize_chunk(chunk) for chunk in chunks]
        combined = " ".join(summaries)

    inputs = tokenizer(
        combined,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=200,
            min_length=100,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [15]:
text = train[0]["article"]

chunks = chunk_text(text)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    decoded = tokenizer.decode(chunk, skip_special_tokens=True)
    print(f"\nCHUNK {i + 1}")
    print(decoded[:500])
    

Number of chunks: 8

CHUNK 1
arp 220 is the nearest ( [MATH] 77 mpc ) example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels. it contains two nuclei separated by 350 pc, both surrounded by massive discs of dense molecular gas ( e.g., *; *; *; *; radio detections of supernovae at a rate of 13 yr [MATH] confirm that huge populations of massive stars are present with an implied star formation rate ( sfr ) of [MATH] yr [MATH]. although arp 220 could contain active galactic nuclei ( 

CHUNK 2
.3 & 2.1 2.3 & + + + + + + + + + + in this paper, we study cosmic ray interactions in the arp 220 starburst nuclear regions using an updated version of the models, hereafter yegz. we develop a model with two spatial zones to accurately represent the inner and outer regions of the western nucleus as defined by its molecular gas properties. we incorporate photopion energy losses and photon photon interactions to account for the extreme fir radiation field. we

In [39]:
summary = summarize_long_document(train[0]["article"])

print("BART SUMMARY:")
print(summary)

print("\n" + "="*80 + "\n")

print("REFERENCE ABSTRACT:")
print(train[0]["abstract"])

BART SUMMARY:
arp 220 is the nearest example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels. it contains two nuclei separated by 350 pc, both surrounded by massive discs of dense molecular gas. radio detections of supernovae at a rate of 13 yr [MATH] confirm that huge populations of massive stars are present. The nuclei of arp 220 are of particular interest as they contain more than half of the total bolometric infrared luminosity of the galaxy.


REFERENCE ABSTRACT:
the cores of arp 220, the closest ultraluminous infrared starburst galaxy, provide an opportunity to study interactions of cosmic rays under extreme conditions. in this paper, we model the populations of cosmic rays produced by supernovae in the central molecular zones of both starburst nuclei. we find that [MATH] of cosmic rays are absorbed in these regions due to their huge molecular gas contents, and thus, the nuclei of arp 220 nearly complete proton calorimeters. as the cos

In [36]:
def filter_reference_chunks(chunks):
    filtered = []

    for chunk in chunks:
        text = tokenizer.decode(chunk, skip_special_tokens=True)
        years = re.findall(r"\b(?:19|20)\d{2}[a-z]?\b", text)

        if len(years) >= 8:
            continue

        filtered.append(chunk)

    return filtered

In [37]:
text = train[0]["article"]

chunks = chunk_text(text)
filtered_chunks = filter_reference_chunks(chunks)

print("Original chunks:", len(chunks))
print("After filtering:", len(filtered_chunks))

for i, chunk in enumerate(filtered_chunks):
    decoded = tokenizer.decode(chunk, skip_special_tokens=True)
    print(f"\nCHUNK {i + 1}")
    print(decoded[:300])

Original chunks: 8
After filtering: 6

CHUNK 1
arp 220 is the nearest ( [MATH] 77 mpc ) example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels. it contains two nuclei separated by 350 pc, both surrounded by massive discs of dense molecular gas ( e.g., *; *; *; *; radio detections of supernovae at a r

CHUNK 2
.3 & 2.1 2.3 & + + + + + + + + + + in this paper, we study cosmic ray interactions in the arp 220 starburst nuclear regions using an updated version of the models, hereafter yegz. we develop a model with two spatial zones to accurately represent the inner and outer regions of the western nucleus as 

CHUNK 3
 secondary cosmic rays, we use our calculations of the population of energetic particles to predict the radio, [MATH] -ray, and neutrino spectra. for the [MATH] -ray spectrum, we include both leptonic ( bremsstrahlung, inverse compton ) and hadronic ( neutral pion decay ) emission mechanisms. for th

CHUNK 4
 radio emission is ther

In [18]:
import re

In [35]:
text = train[0]["article"]

text = remove_references(text)

print(text[-2000:])

sely, we see a much larger change in the ratio of magnetic field energy density to cosmic ray energy density. because the cosmic ray energy density depends on the particle energy loss rate, it does not increase at the same rate as the magnetic and radiation field energy densities ( yoast - hull, gallagher, zweibel, in preparation ). thus, the magnetic fields exceed energy equipartition with the cosmic rays by more than two orders of magnitude ( see table 3 ). this work was supported in part by nsf ast-0907837, nsf phy-0821899 ( to the center for magnetic self - organization in laboratory and astrophysical plasmas ), and nsf phy-0969061 ( to the icecube collaboration ). part of this research was carried out during jsg s appointment as a jubileumsprofessor at the chalmers university of technology. we thank susanne aalto, kazushi sakamoto, dave sanders, nick scoville, and eskil varenius for conversations on arp 220, justin vandenbroucke and reinhard schlickeiser for discussions regarding 

In [41]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

In [43]:
import numpy as np

In [44]:
bart_results = []

for i in range(10):
    article = test[i]["article"]
    reference = test[i]["abstract"]

    summary = summarize_long_document(article)

    scores = scorer.score(reference, summary)

    bart_results.append({
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure
    })

print("ROUGE-1:", np.mean([x["rouge1"] for x in bart_results]))
print("ROUGE-2:", np.mean([x["rouge2"] for x in bart_results]))
print("ROUGE-L:", np.mean([x["rougeL"] for x in bart_results]))

ROUGE-1: 0.3697205240533734
ROUGE-2: 0.12554725415311546
ROUGE-L: 0.21000081892769723
